In [1]:
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

In [2]:
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results
# Load the model
model = build_sam3_image_model()
processor = Sam3Processor(model)

E:\Uni\Bachelorarbeit\sam3\sam3\model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
#your_path/folder
folder_path = "dataset/mvtec_anomaly_detection/hazelnut/test/print"

ground_truth_path = "dataset/mvtec_anomaly_detection/hazelnut/ground_truth/print"
#.png .jpg .gif etc.
#can be empty string "" if different image types and no non-image files are in the folder
image_type = ".png"

In [4]:
from PIL import Image
import glob

folder_path = folder_path + "/*" + image_type

image_list = []
image_paths = []
for filename in sorted(glob.glob(folder_path)):
    im = Image.open(filename)
    image_list.append(im)
    image_paths.append(filename)
   


In [5]:
prompt_list = ["white color"]
prompt = "white color"
confidence_threshold = 0.5

In [6]:
import matplotlib.pyplot as plt

masks = []
single_masks = []
boxes = [] 
scores = []
labels = []
maps = []

for image in image_list:
    inference_state = processor.set_image(image)
    #for prompt in prompt_list:
    processor.reset_all_prompts(inference_state)
    inference_state = processor.set_confidence_threshold(state=inference_state, threshold=confidence_threshold)
    inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
    print(inference_state["masks"].shape)
    masks.append(inference_state["masks"])
    if(inference_state["masks"].size(dim=0) > 1):
        single_masks.append(torch.max(inference_state["masks"].to(torch.float), dim=0).values)
        maps.append(torch.max(inference_state["masks_logits"], dim=0).values)
        labels.append(torch.tensor(1, dtype=int).to(device="cuda"))
        
    elif (inference_state["masks"].size(dim=0) == 1):
        single_masks.append(inference_state["masks"])
        maps.append(inference_state["masks_logits"].squeeze(0))
        labels.append(torch.tensor(1, dtype=int).to(device="cuda"))
        
    else:
        single_masks.append(inference_state["masks"])
        maps.append(torch.zeros(1,1024,1024).to(device="cuda"))
        labels.append(torch.tensor(0, dtype=int).to(device="cuda"))
    boxes.append(inference_state["boxes"])
    scores.append(inference_state["scores"])
    

    

torch.Size([1, 1, 1024, 1024])
torch.Size([1, 1, 1024, 1024])
torch.Size([5, 1, 1024, 1024])
torch.Size([2, 1, 1024, 1024])
torch.Size([1, 1, 1024, 1024])
torch.Size([0, 1, 1024, 1024])
torch.Size([2, 1, 1024, 1024])
torch.Size([0, 1, 1024, 1024])
torch.Size([4, 1, 1024, 1024])
torch.Size([3, 1, 1024, 1024])
torch.Size([6, 1, 1024, 1024])
torch.Size([0, 1, 1024, 1024])
torch.Size([0, 1, 1024, 1024])
torch.Size([1, 1, 1024, 1024])
torch.Size([0, 1, 1024, 1024])
torch.Size([0, 1, 1024, 1024])
torch.Size([0, 1, 1024, 1024])


In [7]:
ground_truth_path = ground_truth_path + "/*" + image_type

gt_mask_list = []
gt_mask_paths = []
for filename in sorted(glob.glob(ground_truth_path)):
    im = Image.open(filename)
    gt_mask_list.append(im)
    gt_mask_paths.append(filename)


In [8]:
#enter the indices of good pictures (without defects) here
good = []

In [9]:
from anomalib.metrics import AUROC
from anomalib.data.dataclasses.torch import ImageBatch
from torchvision import transforms
transform = transforms.ToTensor()
auroc = AUROC(fields=["anomaly_map", "gt_mask"])
images_as_tensors = []

for image in image_list:
    images_as_tensors.append(transform(image))

images_tensor = torch.stack(images_as_tensors).to(device="cuda")

gt_masks_as_tensors = []

for image in gt_mask_list:
    gt_masks_as_tensors.append(transform(image))

#print(gt_masks_as_tensors[0])

gt_masks_tensor = torch.stack(gt_masks_as_tensors).to(device="cuda")

batch = ImageBatch(
    image=images_tensor,
    gt_label=torch.ones(len(image_list), dtype=int).to(device="cuda"),
    image_path=image_paths,
    pred_label=torch.stack(labels, dim=0),
    #pred_mask=,
    #pred_score=scores,
    gt_mask=gt_masks_tensor,
    mask_path=gt_mask_paths,
    anomaly_map=torch.stack(maps, dim=0)
)

for item in batch:
    auroc.update(item)
print(auroc.compute())


tensor(0.6457, device='cuda:0')
